# Spectral democracy as a candidate mechanism for Muon's advantage

## A 3-layer 4x4 deep-linear toy-model companion to `spectral_democracy_test.py`

This notebook is the explanatory counterpart to:

- `experiments/Tier_1_Core_Mechanism_Tests/SPECTRAL_DEMOCRACY_test/spectral_democracy_test.py`

It deliberately stays within the same toy experimental scope as the script and is intended to **use the script's implementation directly**, rather than reimplementing a divergent experiment in notebook cells.

## Framing and scope

**Question.** In this 3-layer 4x4 deep-linear toy model, is Muon's advantage plausibly associated with a more "democratic" distribution of update-direction mass across the current Hessian eigenbasis?

**Implemented proxy.** The script measures a democracy proxy
\[
D(d; V_H) = \frac{p_{10}(|V_H^\top d|)}{p_{90}(|V_H^\top d|)},
\]
where \(d\) is the update direction and \(V_H\) are Hessian eigenvectors. Larger values mean the absolute projection magnitudes are less concentrated by this **particular proxy**. This is not a direct proof of perfect uniformity.

**Compared methods.**

1. **SGD** — raw gradient direction.
2. **Muon** — per-layer polar-factor update direction.
3. **DemocraticSGD** — oracle control: equalize gradient projection magnitudes in the **current Hessian eigenbasis**, then rescale to the gradient norm.
4. **RandomDemocratic** — same equalization idea, but in a random orthogonal basis.

**Claim calibration.** The goal here is not to prove that spectral democracy is *the* general or exclusive mechanism behind Muon. The goal is narrower: evaluate whether this proxy and these controls provide **candidate-mechanism evidence in this toy setting**.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np

NOTEBOOK_SMOKE_MODE = os.environ.get("SPECTRAL_DEMOCRACY_NOTEBOOK_SMOKE", "0") == "1"

import matplotlib
if NOTEBOOK_SMOKE_MODE:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from IPython.display import Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)


def render_figure(fig):
    if NOTEBOOK_SMOKE_MODE:
        plt.close(fig)
    else:
        plt.show()


def resolve_experiment_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.append(base / "experiments" / "Tier_1_Core_Mechanism_Tests" / "SPECTRAL_DEMOCRACY_test")
        candidates.append(base)
    for candidate in candidates:
        script_path = candidate / "spectral_democracy_test.py"
        if script_path.exists():
            return candidate
    raise FileNotFoundError("Could not locate spectral_democracy_test.py from the current notebook working directory.")


EXPERIMENT_DIR = resolve_experiment_dir()
SCRIPT_PATH = EXPERIMENT_DIR / "spectral_democracy_test.py"
if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

import spectral_democracy_test as sdt

print(f"Notebook smoke mode: {NOTEBOOK_SMOKE_MODE}")
print(f"Counterpart script: {SCRIPT_PATH}")
print(f"Resolved experiment directory: {EXPERIMENT_DIR}")

## Reproducibility and runtime notes

This first completion pass preserves the current toy experiment as closely as possible:

- 3 layers, 4x4 matrices, 48 parameters total
- 500 optimization steps by default
- Hessian recomputed every 50 steps
- 5 seeds by default
- same four methods
- same per-method learning-rate sweep concept
- same toy-model finite-difference full-Hessian setup

The main correctness change versus the earlier implementation is that the loss trajectory now has **explicit post-update semantics**:

- `losses[0]` = initial loss before any update
- `losses[t+1]` = post-update loss after step `t`
- `final_loss == losses[-1]`

The notebook can be smoke-executed headlessly by setting:

- `SPECTRAL_DEMOCRACY_NOTEBOOK_SMOKE=1`

In smoke mode it uses a reduced config to keep automated validation manageable.

In [ ]:
def make_notebook_config():
    if NOTEBOOK_SMOKE_MODE:
        return {
            "n_steps": 40,
            "n_seeds": 2,
            "hessian_recompute_every": 10,
            "lr_candidates": np.array([1e-3, 5e-3]),
            "loss_sample_stride": 10,
            "democracy_sample_stride": 10,
            "scope_note": (
                "SMOKE MODE ONLY: reduced toy-model run used solely to validate notebook code paths. "
                "Do not interpret these outputs as the default experiment."
            ),
        }
    return None


NOTEBOOK_CONFIG = make_notebook_config()
RESULTS = sdt.run_experiment(config=NOTEBOOK_CONFIG, verbose=not NOTEBOOK_SMOKE_MODE)
CHECKS = sdt.run_basic_consistency_checks(RESULTS)
CONFIG = RESULTS["config"]
WORKLOAD = RESULTS["workload_estimate"]
METHODS = RESULTS["methods"]
SEED_RESULTS = RESULTS["seed_results"]
AGG = RESULTS["aggregate"]
REP = RESULTS["representative"]

print("Consistency checks:", CHECKS)
print("Effective scope note:", RESULTS["scope_note"])
print("Seeds:", [row["seed"] for row in AGG["seed_summary_rows"]])
print("Approx workload:", WORKLOAD)

In [ ]:
def show_table(rows, index=None, precision=4):
    if pd is not None:
        df = pd.DataFrame(rows)
        if index is not None and index in df.columns:
            df = df.set_index(index)
        display(df.round(precision))
    else:
        for row in rows:
            print(row)


def stack_metric(method, key):
    return np.stack([seed_result["method_runs"][method][key] for seed_result in SEED_RESULTS], axis=0)


def mean_and_std(arr):
    return np.mean(arr, axis=0), np.std(arr, axis=0)


def bootstrap_mean_ci(values, rng_seed=0, n_boot=2000, alpha=0.05):
    values = np.asarray(values, dtype=float)
    rng = np.random.RandomState(rng_seed)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boots.append(np.mean(sample))
    low, high = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(np.mean(values)), float(low), float(high)


def loss_summary_frame_rows():
    rows = []
    for row in AGG["summary_table_rows"]:
        rows.append({
            "method": row["method"],
            "mean_final_loss": row["mean_final_loss"],
            "std_final_loss": row["std_final_loss"],
            "mean_democracy_proxy": row["mean_democracy"],
            "std_democracy_proxy": row["std_democracy"],
        })
    return rows

## Effective configuration used in this run

This cell logs the exact configuration and workload estimate returned by the shared script implementation.

In [ ]:
config_rows = [
    {"field": "experiment_name", "value": CONFIG["experiment_name"]},
    {"field": "scope_note", "value": CONFIG["scope_note"]},
    {"field": "dim", "value": CONFIG["dim"]},
    {"field": "n_layers", "value": CONFIG["n_layers"]},
    {"field": "n_params", "value": CONFIG["n_params"]},
    {"field": "n_steps", "value": CONFIG["n_steps"]},
    {"field": "hessian_recompute_every", "value": CONFIG["hessian_recompute_every"]},
    {"field": "n_seeds", "value": CONFIG["n_seeds"]},
    {"field": "seeds", "value": sdt.build_seed_list(CONFIG)},
    {"field": "target_singular_values", "value": CONFIG["target_singular_values"].tolist()},
    {"field": "lr_candidates", "value": CONFIG["lr_candidates"].tolist()},
    {"field": "random_basis_seed_policy", "value": CONFIG["random_basis_seed_policy"]},
    {"field": "loss_trajectory_semantics", "value": RESULTS["notes"]["loss_trajectory_semantics"]},
    {"field": "democracy_trajectory_semantics", "value": RESULTS["notes"]["democracy_trajectory_semantics"]},
]
show_table(config_rows)

workload_rows = [{"field": k, "value": v} for k, v in WORKLOAD.items()]
show_table(workload_rows)

## Aggregate summary table

These are seed-aggregated summaries returned by the script, now using the corrected terminal-loss definition.

In [ ]:
show_table(loss_summary_frame_rows(), index="method", precision=6)

### Interpretation

- Lower final loss is better.
- Higher democracy proxy means the chosen update direction is less concentrated by the script's `p10/p90` projection proxy.
- Because this is only 5 seeds by default, the spread is informative but still limited.

## Seed-level results: final losses, recoveries, and tuned learning rates

The table below exposes the per-seed raw outcomes needed to audit how the aggregate summaries were formed.

In [ ]:
show_table(AGG["seed_summary_rows"], precision=6)

## Figure 1 — Mean ± standard deviation loss trajectories across seeds

The plot below uses the **shared script trajectories**. Because `losses[0]` is the initial loss and `losses[-1]` is the terminal post-training loss, the x-axis now has an unambiguous interpretation.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for method in METHODS:
    losses = stack_metric(method, "losses")
    mean_loss, std_loss = mean_and_std(losses)
    steps = np.arange(losses.shape[1])
    ax.plot(steps, mean_loss, label=method)
    ax.fill_between(steps, mean_loss - std_loss, mean_loss + std_loss, alpha=0.2)
ax.set_xlabel("Trajectory index (0 = initial loss, t+1 = post-update loss after step t)")
ax.set_ylabel("Loss")
ax.set_title("Mean ± std loss trajectories across seeds")
ax.legend()
ax.grid(alpha=0.3)
render_figure(fig)

### Loss-trajectory interpretation

If Muon and/or DemocraticSGD remain consistently below SGD over the trajectory, that supports the descriptive claim that the final-loss differences are not merely a late-step artifact. It still does **not** prove a unique mechanism.

## Figure 2 — Mean ± standard deviation democracy-proxy trajectories

These trajectories summarize the update-direction democracy proxy for all four methods. The metric is meaningful as a descriptive proxy, but should not be read as a direct measure of exact uniformity.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for method in METHODS:
    democracy = stack_metric(method, "democracy_ratios")
    mean_dem, std_dem = mean_and_std(democracy)
    steps = np.arange(democracy.shape[1])
    ax.plot(steps, mean_dem, label=method)
    ax.fill_between(steps, mean_dem - std_dem, mean_dem + std_dem, alpha=0.2)
ax.set_xlabel("Optimization step")
ax.set_ylabel("Democracy proxy (p10/p90)")
ax.set_title("Mean ± std democracy-proxy trajectories across seeds")
ax.legend()
ax.grid(alpha=0.3)
render_figure(fig)

### Democracy-proxy interpretation

A sustained Muon-above-SGD gap in this proxy is consistent with the spectral-democracy story. However, the proxy only captures the spread of absolute projection magnitudes in the current Hessian eigenbasis; it is not itself a full mechanistic proof.

## Figure 3 — Per-seed final losses and recovery comparison

This section checks whether the ordering seen in the averages is stable seed by seed, and whether the Hessian-basis control tends to recover more of Muon's advantage than the random-basis control.

In [ ]:
seed_rows = AGG["seed_summary_rows"]
seeds = [row["seed"] for row in seed_rows]
sgd_losses = np.array([row["SGD_final_loss"] for row in seed_rows])
muon_losses = np.array([row["Muon_final_loss"] for row in seed_rows])
dem_losses = np.array([row["DemocraticSGD_final_loss"] for row in seed_rows])
rnd_losses = np.array([row["RandomDemocratic_final_loss"] for row in seed_rows])
dem_rec = np.array([row["Democratic_recovery_percent"] for row in seed_rows])
rnd_rec = np.array([row["Random_recovery_percent"] for row in seed_rows])

x = np.arange(len(seeds))
width = 0.2

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(x - 1.5 * width, sgd_losses, width=width, label="SGD")
axes[0].bar(x - 0.5 * width, muon_losses, width=width, label="Muon")
axes[0].bar(x + 0.5 * width, dem_losses, width=width, label="DemocraticSGD")
axes[0].bar(x + 1.5 * width, rnd_losses, width=width, label="RandomDemocratic")
axes[0].set_xticks(x)
axes[0].set_xticklabels(seeds)
axes[0].set_xlabel("Seed")
axes[0].set_ylabel("Final loss")
axes[0].set_title("Per-seed final losses")
axes[0].legend(fontsize=8)
axes[0].grid(axis="y", alpha=0.3)

axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].bar(x - 0.2, dem_rec, width=0.4, label="DemocraticSGD recovery")
axes[1].bar(x + 0.2, rnd_rec, width=0.4, label="RandomDemocratic recovery")
axes[1].set_xticks(x)
axes[1].set_xticklabels(seeds)
axes[1].set_xlabel("Seed")
axes[1].set_ylabel("Recovery of Muon's advantage (%)")
axes[1].set_title("Per-seed recovery comparison")
axes[1].legend(fontsize=8)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
render_figure(fig)

### Final-loss / recovery interpretation

Recovery is defined relative to the seed-specific SGD-to-Muon final-loss gap. Values above 100% mean the oracle Hessian-basis control outperformed Muon on that seed under its own tuned learning rate; this is informative but should not be over-read as a clean causal isolation of one mechanism.

## T1/T2/T3 heuristic verdicts and uncertainty summaries

The script preserves the original threshold-style summaries, but this notebook explicitly treats them as **heuristic diagnostics**, not formal mechanism proof.

In [ ]:
tests = AGG["tests"]
verdict_rows = [
    {
        "test": "T1",
        "description": tests["T1"]["description"],
        "value": tests["T1"]["value"],
        "threshold": tests["T1"]["threshold"],
        "passed": tests["T1"]["passed"],
        "caveat": tests["T1"]["caveat"],
    },
    {
        "test": "T2",
        "description": tests["T2"]["description"],
        "value": tests["T2"]["value"],
        "threshold": tests["T2"]["threshold"],
        "passed": tests["T2"]["passed"],
        "caveat": tests["T2"]["caveat"],
    },
    {
        "test": "T3",
        "description": tests["T3"]["description"],
        "value": tests["T3"]["fraction"],
        "threshold": tests["T3"]["threshold_fraction"],
        "passed": tests["T3"]["passed"],
        "caveat": tests["T3"]["caveat"],
    },
]
show_table(verdict_rows, index="test", precision=6)

bootstrap_rows = []
for label, values in [
    ("Mean Muon/SGD democracy-ratio multiplier", AGG["d_ratios_muon_over_sgd"]),
    ("Mean DemocraticSGD recovery (%)", AGG["recoveries_dem"]),
    ("Mean RandomDemocratic recovery (%)", AGG["recoveries_rnd"]),
    ("Mean Muon final-loss improvement over SGD", np.array([row["SGD_final_loss"] - row["Muon_final_loss"] for row in seed_rows])),
    ("Mean DemocraticSGD minus RandomDemocratic recovery (%)", dem_rec - rnd_rec),
]:
    mean_value, ci_low, ci_high = bootstrap_mean_ci(values)
    bootstrap_rows.append({
        "quantity": label,
        "mean": mean_value,
        "bootstrap_95pct_CI_low": ci_low,
        "bootstrap_95pct_CI_high": ci_high,
    })
show_table(bootstrap_rows, precision=6)

print("Overall heuristic T1/T2/T3 verdict:", "ALL THREE PASS" if tests["overall_pass"] else "SOME TESTS FAIL")
print("Heuristic note:", tests["note"])

### T1/T2/T3 interpretation

- **T1** asks whether Muon is substantially more democratic than SGD by the chosen proxy.
- **T2** asks whether oracle Hessian-basis equalization recovers a large fraction of Muon's advantage.
- **T3** asks whether random-basis equalization is usually weaker than Hessian-basis equalization.

Even when these pass, the evidence remains limited to this toy model, these seeds, these tuned learning rates, and this exact oracle construction.

## Representative-seed sampled trajectories

To keep the printed reporting in the script and the notebook aligned, the script also returns representative sampled tables.

In [ ]:
print(f"Representative seed index: {REP['seed_index']}, seed value: {REP['seed']}")
show_table(REP["loss_table_rows"], precision=6)
show_table(REP["democracy_table_rows"], precision=6)

## Energy-distribution and basis-projection summary at initialization

The script's representative-seed analysis now returns a careful initial-step summary rather than relying on leaked loop state. The quantities below describe how concentrated each initial direction is in the representative seed's Hessian eigenbasis.

In [ ]:
energy = REP["energy_distribution"]
energy_rows = []
for method in METHODS:
    topk = energy["topk_percent"][method]
    quartiles = energy["quartile_energy_percent"][method]
    energy_rows.append({
        "method": method,
        "top1_percent": topk["top1"],
        "top3_percent": topk["top3"],
        "top10_percent": topk["top10"],
        **quartiles,
        "initial_democracy_proxy": energy["initial_direction_democracy"][method],
    })
show_table(energy_rows, index="method", precision=6)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
quartile_labels = ["Q1_smallest_eigs", "Q2", "Q3", "Q4_largest_eigs"]
x = np.arange(len(quartile_labels))
width = 0.18
for i, method in enumerate(METHODS):
    values = [energy["quartile_energy_percent"][method][label] for label in quartile_labels]
    axes[0].bar(x + (i - 1.5) * width, values, width=width, label=method)
axes[0].set_xticks(x)
axes[0].set_xticklabels(quartile_labels, rotation=15)
axes[0].set_ylabel("Percent of squared projection energy")
axes[0].set_title("Initial energy across Hessian-eigenvalue quartiles")
axes[0].legend(fontsize=8)
axes[0].grid(axis="y", alpha=0.3)

for i, method in enumerate(METHODS):
    topk = energy["topk_percent"][method]
    axes[1].bar(np.array([0, 1, 2]) + (i - 1.5) * width, [topk["top1"], topk["top3"], topk["top10"]], width=width, label=method)
axes[1].set_xticks([0, 1, 2])
axes[1].set_xticklabels(["top1", "top3", "top10"])
axes[1].set_ylabel("Percent of squared projection energy")
axes[1].set_title("Initial concentration in top projected directions")
axes[1].legend(fontsize=8)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
render_figure(fig)

print("Representative-seed Hessian spectrum summary:")
print("  smallest5:", np.round(energy["hessian_spectrum"]["smallest5"], 3))
print("  largest5 :", np.round(energy["hessian_spectrum"]["largest5"], 3))
print("  condition estimate:", round(energy["hessian_spectrum"]["condition_estimate"], 3))

### Energy-summary interpretation

Lower top-1 / top-3 concentration and a less extreme quartile concentration pattern are consistent with the update direction being less spectrally concentrated in the Hessian eigenbasis. This remains a descriptive diagnostic, not a stand-alone proof of mechanism.

## Calibrated conclusion and limitations

This notebook and the companion script now agree on the implemented experiment and on the loss-trajectory semantics. The evidence produced here should be read as follows:

1. **What is supported:** in this 3-layer 4x4 deep-linear toy model, Muon often shows a higher democracy proxy than SGD, and oracle Hessian-basis equalization can recover part of Muon's advantage.
2. **What is not supported:** the current implementation does not establish spectral democracy as the unique or general mechanism behind Muon, nor does it justify broad claims beyond this toy family.
3. **Why the evidence is still useful:** the Hessian-basis control and the random-basis control make the toy mechanism story more concrete and falsifiable than a purely narrative claim.
4. **Main limitations:** exact finite-difference Hessians, small state dimension, 5 seeds by default, heuristic thresholds, and a fixed random-basis seed policy unless the config is changed explicitly.

In [ ]:
muon_ratio_mean = AGG["mean_d_ratio_muon_over_sgd"]
dem_rec_mean = AGG["mean_recovery_dem"]
rnd_rec_mean = AGG["mean_recovery_rnd"]
verdict = "all three heuristic tests pass" if AGG["tests"]["overall_pass"] else "not all heuristic tests pass"

summary_md = f"""
### Run-specific bottom line

- Mean Muon/SGD democracy-ratio multiplier: **{muon_ratio_mean:.2f}x**
- Mean DemocraticSGD recovery: **{dem_rec_mean:.1f}%**
- Mean RandomDemocratic recovery: **{rnd_rec_mean:.1f}%**
- Heuristic T1/T2/T3 status: **{verdict}**

**Calibrated reading:** this run provides candidate-mechanism evidence for the spectral-democracy story **within this toy setting only**.
"""
display(Markdown(summary_md))